# 中心链：内部接触、Rg与外部连接的联合分析

读取按simulation_id、source_frame、chain_id配对的结果，不重算接触或Rg。
浅点为同帧同链，菱形为帧内均值；条件差异与单一模拟内部的关系需分别解释，相关不等于竞争或因果。
中心链可逐帧换身份，不能把它当作固定链的连续变化。内部每残基接触数=2E/N，外部=Ca/N。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project.py").is_file())
sys.path.insert(0, str(ROOT))
import project
import result_files as files

SELECTED = None
OUTPUT = ROOT / "02_calculate/03_contacts/01_central_chain/03_joint_analysis/output"
SIMULATIONS = project.select_simulations(ROOT / "simulations.yaml", SELECTED)
RESULTS = {s["id"]: files.completed(OUTPUT / s["id"]) for s in SIMULATIONS}
SOURCES = {name: files.read_json(path / "metadata.json") for name, path in RESULTS.items()}
COLORS = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(RESULTS)}
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.frameon": False,
    }
)


def export(fig, name):
    for key, path in RESULTS.items():
        files.completed(path)
        if files.read_json(path / "metadata.json") != SOURCES[key]:
            raise ValueError("Results changed; restart notebook")
    for ext in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    files.write_json(FIGURES / f"{name}.sources.json", SOURCES)


In [ ]:
PAIRED = {name: pd.read_csv(path / "paired_chain_frames.csv.gz") for name, path in RESULTS.items()}
FRAME = {name: pd.read_csv(path / "frame_means.csv") for name, path in RESULTS.items()}
shared = set.intersection(*(set(data.threshold.unique()) for data in FRAME.values()))
PRIMARY_THRESHOLD = min(shared)
comparisons = [
    ("contact_count", "total_contacts", "Internal contact pairs", "External residue contacts"),
    ("contact_count", "partner_count", "Internal contact pairs", "Partner chains"),
    ("rg_nm", "partner_count", "Mass-weighted Rg (nm)", "Partner chains"),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
for ax, (x, y, xlabel, ylabel) in zip(axes, comparisons):
    for name in RESULTS:
        data = PAIRED[name]
        data = data[data.threshold.eq(PRIMARY_THRESHOLD)]
        frame = FRAME[name]
        frame = frame[frame.threshold.eq(PRIMARY_THRESHOLD)]
        ax.scatter(
            data[x],
            data[y],
            s=5,
            alpha=0.04,
            color=COLORS[name],
            edgecolors="none",
            rasterized=True,
        )
        ax.scatter(
            frame[x],
            frame[y],
            s=13,
            alpha=0.5,
            color=COLORS[name],
            marker="D",
            label=name,
            edgecolors="none",
        )
    ax.set(xlabel=xlabel, ylabel=ylabel)
axes[0].legend(title="Diamonds: frame means")
fig.suptitle(f"Partner threshold ≥ {PRIMARY_THRESHOLD} residue contacts", fontsize=10)
export(fig, "intra_inter_rg_relationships")
plt.show()
display(
    pd.concat([pd.read_csv(path / "summary.csv") for path in RESULTS.values()], ignore_index=True)
)
